# End-to-end classification: a rare-event churn flag

This is the first of three notebooks that take the same problem all the way from raw
data to a signed-off, serviceable model. Classification is the task here; the
companions cover [regression](02_regression_end_to_end.ipynb) and
[time series](03_time_series_end_to_end.ipynb).

The target is a **rare event**: a subscription is flagged high-risk for churn.
About 12 % of accounts carry the flag, so accuracy is a bad headline metric and the
interesting questions are about ranking (lift, cumulative gain), calibration, and
whether the model beats a trivial rule at all.

## What you will see

| Stage | SKSurrogate pieces |
| --- | --- |
| Profiling, missingness, association and stability diagnostics | `dataset_profile`, `api.eda`, `api.preprocessing` |
| Choosing an imputation method by measurement, not by habit | `api.preprocessing.impute_partition`, scikit-learn imputers, `FoldLocalDataPreprocessor` |
| Bounded pipeline search | `AML(preset="classification")`, `to_audit_dict()` |
| Evaluation with uncertainty | `mltrack.nested_cv_evaluation` |
| Explanation, ranking and calibration charts | `mltrack.permutation_importance`, `plot_*` helpers |
| Packaging, quality gate, serving | `save_bundle`, `ModelRegistry`, `DeploymentApprovalGate`, `predict_batch` |
| Report | Markdown + CSV/JSON + PNG, and a branded PDF via `api.reports` |

## How to run

From the repository root, with the project environment active:

```bash
jupyter lab examples/notebooks/01_classification_end_to_end.ipynb
```

The notebook is offline by default — the data comes from
`examples/notebooks/notebook_data.py`, a seeded generator with the structure that
makes this workflow necessary (non-linearity, an interaction, mixed dtypes, a
perfectly collinear duplicate, injected MAR and MCAR gaps). For a real dataset:

```bash
SKSURROGATE_NOTEBOOK_DATA=openml SKSURROGATE_OPENML_KEY=... \
    jupyter lab examples/notebooks/01_classification_end_to_end.ipynb
```

which loads OpenML `credit-g` instead. Everything downstream is unchanged; only the
provenance table and the dtypes differ.

Artifacts land in `classification-report/` in the working directory. The notebook
does not save rendered cell output: the `.ipynb` you are reading is the source.

**This is a methods demonstration, not a scientific result.** The data is synthetic,
the search budget is deliberately small, and the confidence intervals are honest
about being wide.

In [ ]:
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The example data generators and report helpers live next to this notebook.
REPO_ROOT = Path.cwd()
NOTEBOOK_DIR = REPO_ROOT / "examples" / "notebooks"
if not (NOTEBOOK_DIR / "notebook_data.py").exists():
    NOTEBOOK_DIR = REPO_ROOT  # launched from inside examples/notebooks
sys.path.insert(0, str(NOTEBOOK_DIR.resolve()))

import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (opt-in import)
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import SKSurrogate
from SKSurrogate import (
    DeploymentApprovalGate,
    ModelBundle,
    FoldLocalDataPreprocessor,
    ModelRegistry,
    assert_bundle_quality,
    dataset_profile,
    drift_report,
    fairness_report,
    load_bundle,
    mltrack,
    predict_batch,
    save_bundle,
    subgroup_performance_report,
)
from api.eda import effect_sizes, formal_tests, missing_mechanism, quality_flags, stability
from api.preprocessing import impute_partition, missing_overview, target_balance
from notebook_data import SEED, SOURCE, artifact_dir, classification_data
from notebook_report import compile_pdf, figure_of, figure_pdf, markdown_table, save_figure

warnings.filterwarnings("ignore", category=FutureWarning)
# The tracker's explanation and plotting paths hand numpy arrays to a model that was
# fitted on numpy arrays; scikit-learn's "X has feature names" note is expected here.
warnings.filterwarnings("ignore", message="X has feature names")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

REPORT_DIR = artifact_dir("classification-report")
print(f"SKSurrogate {SKSurrogate.__version__} from {Path(SKSurrogate.__file__).parent}")
print(f"scikit-learn {sklearn.__version__} | pandas {pd.__version__} | numpy {np.__version__}")
print(f"seed={SEED}  data source={SOURCE!r}  artifacts -> {REPORT_DIR.resolve()}")

## 1. The data, and where it came from

A provenance table is the cheapest reproducibility artefact there is: it says which
generator, which seed, how many rows, and what kind of missingness was injected. Print
it before you look at any result.

In [ ]:
dataset = classification_data()
frame = dataset.frame
TARGET = dataset.target

provenance = pd.DataFrame(
    [
        ("source", dataset.source),
        ("description", dataset.description),
        ("rows x columns", f"{frame.shape[0]} x {frame.shape[1]}"),
        ("target", TARGET),
        ("positive rate", f"{frame[TARGET].mean():.3f}"),
        ("categorical features", ", ".join(frame.select_dtypes(exclude=[np.number]).columns) or "none"),
        ("missing cells %", f"{frame.isna().to_numpy().mean() * 100:.2f}"),
    ],
    columns=["field", "value"],
)
display(provenance)
for key, value in dataset.notes.items():
    if not key.endswith("_indices"):  # long index lists are noise here
        print(f"{key}: {value}")
display(frame.head())

## 2. Profile the frame before touching a model

`dataset_profile` returns bounded, JSON-safe summaries: per-column distributions,
missingness, target associations, exact-missingness patterns, multicollinearity, and
duplicate detection. Rows are sampled without replacement when the frame is large, and
the sample scope is recorded so no table can be read as covering more than it does.

Read the output as *diagnostics*, not evidence: an association between a feature and
the target is not a mechanism.

In [ ]:
profile = dataset_profile(
    frame,
    target=TARGET,
    group_by="contract",
    max_rows=2000,
    max_features=50,
    max_categories=10,
    random_state=SEED,
)

columns_table = pd.DataFrame(profile["columns"])[
    ["column", "dtype", "kind", "unique_count", "missing_count", "missing_rate"]
]
display(columns_table)
print("sampled rows:", profile["sampled_rows"], "of", profile["rows"])
display(pd.DataFrame(profile["target_associations"]).head(8))

The profile also answers the questions that decide preprocessing rather than modelling:
which features are constant or all-missing, which are exact duplicates, and how badly
the numeric block is conditioned.

In [ ]:
quality = profile["data_quality"]
print("duplicate features :", quality["duplicate_features"])
print("duplicate rows     :", quality["duplicate_rows"])
print("constant features  :", quality["constant_features"])
print("all-missing        :", quality["all_missing_features"])
mcl = profile["multicollinearity"]
print(
    "condition number   :",
    mcl["condition_number"]
    if mcl["condition_number"] is not None
    else f"rank deficient ({mcl['condition_number_reason'].replace('_', ' ')})",
)
print("largest VIF        :", max((row["value"] or 0.0) for row in mcl["vif"]))
print("VIF note           :", mcl["interpretation"])

In [ ]:
# One figure that answers "what does this frame look like?" at a glance.
missing_rate = columns_table.assign(pct=100 * columns_table["missing_rate"]).sort_values("pct")
balance = frame[TARGET].value_counts(normalize=True).sort_index()
associations = (
    pd.DataFrame(profile["target_associations"])
    .assign(strength=lambda d: d["value"].abs())
    .nlargest(10, "strength")
    .sort_values("strength")
)

figure, axes = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
axes[0].barh(missing_rate["column"], missing_rate["pct"], color="C0")
axes[0].set(xlabel="missing cells (% of rows)", title="Missingness by column")
axes[0].set_xlim(0, max(1.0, missing_rate["pct"].max() * 1.15))

axes[1].bar(["no churn", "high risk"], balance.to_numpy(), color=["C0", "C3"])
for index, value in enumerate(balance.to_numpy()):
    axes[1].text(index, value + 0.01, f"{value:.1%}", ha="center")
axes[1].set(ylabel="share of rows", title=f"Target balance ({TARGET})")
axes[1].set_ylim(0, 1.05)

axes[2].barh(associations["feature"], associations["strength"].to_numpy(), color="C2")
axes[2].set(xlabel="|association| with target", title="Top 10 target associations")
figure.suptitle("Classification frame: profile overview", fontsize=13)
save_figure(figure, REPORT_DIR, "eda-overview")
display(figure)

## 3. Missing data: how much, and of what kind?

Two different questions, and conflating them is how imputation goes wrong:

1. **How much is missing, and in what pattern?** (`missing_overview`,
   `profile["missingness_patterns"]`)
2. **Is the missingness *informative*?** If a value is more likely to be missing when
   other observed values are extreme, dropping those rows biases the fit and the
   mechanism matters.

`missing_mechanism` reports Little's omnibus MCAR test plus a per-column shift summary:
the standardised difference between the observed distribution of every other numeric
column when this column is missing versus when it is present.

In [ ]:
overview = missing_overview(frame, target=TARGET, ignored=[])
display(pd.DataFrame([overview]))

patterns = pd.DataFrame(profile["missingness_patterns"]["patterns"])
patterns["missing_features"] = patterns["missing_features"].apply(
    lambda items: ", ".join(items) if items else "(complete)"
)
display(patterns)

mechanism = missing_mechanism(frame, target=TARGET)
print("Little MCAR test:", json.dumps(mechanism["mcar"]))
display(pd.DataFrame(mechanism["shifts"]).sort_values("standardized_difference", ascending=False))
print(mechanism["interpretation"])

Note the `reason: em_did_not_converge`. Little's test is an EM likelihood-ratio test and
needs a non-singular covariance estimate; with the perfectly collinear `spend_ytd` still
in the frame it cannot run. That is the diagnostic telling you about the duplicate before
you have looked for it — section 5 prunes the column and the test then runs.

The shift table works either way, and it is the sharper instrument: `nps_score` is missing
noticeably more often when `late_payments` is high — exactly how the generator built it,
**MAR on `late_payments`**. The practical consequence: an imputer that uses the other
columns can beat a marginal median, and dropping incomplete rows would preferentially
discard high-late-payment accounts.

In [ ]:
shifts = pd.DataFrame(mechanism["shifts"])
nps_shift = shifts[shifts["missing_column"] == "nps_score"].iloc[0]

# Show the mechanism directly: missing rate of nps_score by late-payment count.
by_late = (
    frame.assign(nps_missing=frame["nps_score"].isna())
    .query("late_payments <= 4")
    .groupby("late_payments")["nps_missing"]
    .agg(["mean", "size"])
    .reset_index()
)

figure, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
axes[0].bar(by_late["late_payments"], by_late["mean"], color="C3")
axes[0].set(
    xlabel="late payments in the last year",
    ylabel="share of rows with nps_score missing",
    title="MAR evidence: nps_score misses more often for late payers",
)
for row in by_late.itertuples():
    axes[0].text(row.late_payments, row.mean + 0.004, f"n={row.size}", ha="center", fontsize=8)

shifts_plot = shifts.nlargest(8, "standardized_difference").sort_values("standardized_difference")
axes[1].barh(
    shifts_plot["missing_column"] + " | " + shifts_plot["numeric_column"],
    shifts_plot["standardized_difference"],
    color="C1",
)
axes[1].axvline(0, color="k", linewidth=0.8)
axes[1].axvline(0.2, color="k", linestyle=":", linewidth=0.8)
axes[1].set(
    xlabel="standardised mean difference (observed vs missing rows)",
    title="Which columns separate the missing from the observed",
)
figure.suptitle("Missingness is not silent", fontsize=13)
save_figure(figure, REPORT_DIR, "missing-mechanism")
display(figure)
print(f"nps_score: observed mean {nps_shift['mean_observed']:.2f} vs missing mean "
      f"{nps_shift['mean_missing']:.2f} (standardised difference {nps_shift['standardized_difference']:.2f})")

## 4. Associations, formal tests, effect sizes, stability

Four complementary diagnostics, each answering a different question:

* `formal_tests` — per-column normality tests (Shapiro-Wilk / D'Agostino) with the
  caveat that they are about the *marginal* distribution, not about the model.
* `effect_sizes` — Cohen's d and Cliff's delta between contract groups, with bootstrap
  intervals. A p-value tells you "probably not zero"; a standardised effect tells you
  "probably not worth acting on".
* `stability` — recompute each statistic on two disjoint halves and report the relative
  difference. A statistic that moves by 40 % between halves is not a finding.
* `quality_flags` — identifier-like columns, near-duplicates, quasi-constants, leakage
  suspects.

None of these make a causal claim. All of them are recorded in the report so a reader
can disagree with the modelling choices by inspecting the same numbers.

In [ ]:
tests = formal_tests(frame, target=TARGET, alpha=0.05)
tests_table = pd.DataFrame(tests["columns"]).sort_values("statistic", ascending=False)
display(tests_table)
print(tests["caveat"])

effects = effect_sizes(
    frame, target=TARGET, group_by="contract", bootstrap=200, random_state=SEED
)
effects_table = pd.DataFrame(effects["comparisons"])
display(
    effects_table.assign(abs_d=lambda d: d["cohens_d"].abs())
    .nlargest(8, "abs_d")
    .drop(columns="abs_d")
)
print(effects["interpretation"])

stability_report = stability(frame, target=TARGET, random_state=SEED)
stability_table = pd.DataFrame(stability_report["columns"])
display(stability_table.sort_values("max_relative_difference", ascending=False).head(8))
print("largest half-to-half relative difference:",
      round(stability_report["max_relative_difference"], 4))

## 5. A decision: prune the duplicate and the identifier

`monthly_spend` and `spend_ytd` are the same variable up to a factor of 12. The profile
says so twice: `duplicate_features` names the pair, and the VIF is null with
`perfect_or_near_linear_dependence`.

The profile also shows `billing_code` with 849 distinct values in 900 rows — a per-customer
key that happens to be stored as an integer. It carries no signal, and a tree can memorise
rows with it, so it leaves here too. The rule is shape-based: an integer column where at
least half the rows carry a distinct value is a key, whatever it is called. Compare
`app_logins` (16 levels) or `support_calls` (6 levels), which are genuine measurements.

Keeping both is not harmless: it splits importance between two identical columns,
makes linear models singular, and inflates any weight-based interpretation. It is also
exactly the kind of thing a "drop highly correlated features" step is *supposed* to
catch, so the notebook does it explicitly and records the before/after.

In [ ]:
def describe_condition(number):
    # None means the design matrix was rank deficient (perfect collinearity).
    return "rank deficient" if number is None else f"{number:.3f}"


before_condition = describe_condition(mcl["condition_number"])
def looks_like_identifier(series, row_count):
    # A key is an integer (or string) column whose values are distinct row by row: at
    # least half of the rows carry a new level for integers, practically every row for
    # strings. Continuous float columns are measurements and are never dropped here --
    # a float that is unique on every row is still a measurement.
    if series.dtype.kind in "iu":
        threshold = 0.5
    elif series.dtype == object:
        threshold = 0.98
    else:
        return False
    distinct = series.nunique(dropna=True)
    return distinct > 20 and distinct / row_count > threshold


identifier_like = [
    column
    for column in frame.columns
    if column != TARGET and looks_like_identifier(frame[column], len(frame))
]
print("identifier-like columns:", identifier_like)
frame = frame.drop(columns=["spend_ytd", *identifier_like])
reprofiled = dataset_profile(frame, target=TARGET, max_rows=2000, random_state=SEED)
after_condition = describe_condition(reprofiled["multicollinearity"]["condition_number"])
duplicates_after = reprofiled["data_quality"]["duplicate_features"]

print(f"condition number: {before_condition} -> {after_condition}")
print("duplicates left :", duplicates_after)
print("complete rows   :", reprofiled["data_quality"]["duplicate_rows"]["count"],
      "duplicate rows of", reprofiled["sampled_rows"], "sampled")

# The MCAR test can now estimate the covariance matrix; rerun it on the pruned frame.
mechanism_pruned = missing_mechanism(frame, target=TARGET)
print("Little MCAR test after pruning:", json.dumps(mechanism_pruned["mcar"]))

## 6. Choosing an imputation method by measurement

Rather than picking `median` because it is the default, measure it. The protocol:

1. take the 150 observed `nps_score` values at random and hide them;
2. impute the column with each candidate method;
3. score the reconstruction against the hidden truth.

That isolates imputation quality. `drop_rows` cannot be scored the same way, so it is
reported by the number of rows it destroys instead — for this frame that is a fifth of
the data, which is its own argument.

`knn` is run twice, once raw and once after standardising: nearest-neighbour imputation
compares distances across columns, so an unscaled `spend_ytd`-sized column dominates
the metric. That is a real trap, not a technicality.

In [ ]:
numeric_frame = frame.select_dtypes(include=[np.number])
rng = np.random.default_rng(SEED)
observed_rows = np.flatnonzero(numeric_frame["nps_score"].notna().to_numpy())
held_out = np.sort(rng.choice(observed_rows, size=150, replace=False))
truth = numeric_frame.loc[held_out, "nps_score"].to_numpy()

masked = numeric_frame.copy()
masked.loc[held_out, "nps_score"] = np.nan
column_index = list(numeric_frame.columns).index("nps_score")


def reconstruction_error(imputed_column) -> tuple[float, float]:
    """RMSE and MAE of an imputed nps_score against the hidden values."""
    values = np.asarray(imputed_column)[held_out]
    residual = values - truth
    return float(np.sqrt(np.mean(residual ** 2))), float(np.mean(np.abs(residual)))


rows = []
for method, imputer in (
    ("median", SimpleImputer(strategy="median")),
    ("iterative (MICE-style)", IterativeImputer(max_iter=10, random_state=SEED)),
    ("knn k=5 (raw scale)", KNNImputer(n_neighbors=5)),
):
    filled = imputer.fit_transform(masked)
    rmse, mae = reconstruction_error(filled[:, column_index])
    rows.append({"method": method, "rmse": rmse, "mae": mae, "rows_kept": len(masked)})

# The same kNN on standardised columns, then mapped back to the original units.
scaler = StandardScaler()
scaled = pd.DataFrame(scaler.fit_transform(masked), columns=masked.columns)
filled_scaled = scaler.inverse_transform(KNNImputer(n_neighbors=5).fit_transform(scaled))
rmse, mae = reconstruction_error(filled_scaled[:, column_index])
rows.append({"method": "knn k=5 (standardised)", "rmse": rmse, "mae": mae, "rows_kept": len(masked)})

rows.append(
    {
        "method": "drop_rows",
        "rmse": np.nan,
        "mae": np.nan,
        "rows_kept": int(masked.dropna().shape[0]),
    }
)
imputation_table = pd.DataFrame(rows).sort_values("rmse", na_position="last")
display(imputation_table)

figure, ax = plt.subplots(figsize=(8.4, 4.0), constrained_layout=True)
plotted = imputation_table.dropna(subset=["rmse"])
ax.barh(plotted["method"], plotted["rmse"], color="C0", label="RMSE")
ax.barh(plotted["method"], plotted["mae"], height=0.45, color="C3", label="MAE")
ax.set(xlabel="reconstruction error on 150 hidden nps_score values (lower is better)")
ax.set_title("Imputation quality, measured instead of assumed")
ax.invert_yaxis()
ax.legend()
save_figure(figure, REPORT_DIR, "imputation-benchmark")
display(figure)

Two lessons, both worth carrying into real work:

* **`iterative` wins** because `nps_score` is a noisy function of the other drivers —
  a multivariate conditional model recovers more of it than a marginal statistic.
* **`knn` is scale-sensitive.** Standardising moved it from second-worst to competitive.
  If you report a kNN imputation without saying which scaling you used, the number is
  not reproducible.

The choice is therefore `IterativeImputer`. The value of the experiment is that the
choice is now *evidence*, and the table ships with the report.

## 7. The leakage question nobody asks

`impute_partition` (the Data Preprocessing page in the UI) fits an imputer on a whole
partition and materialises an imputed copy. That is the right tool when the partition
**is** the training set. It is the wrong tool if you then cross-validate on that copy:
the validation fold's own statistics have already been baked into its features.

The safe shape is a scikit-learn transformer inside the pipeline, so every fold fits its
own imputation and its own category vocabulary. `FoldLocalDataPreprocessor` does exactly
that for mixed-type frames — median imputation and scaling for numerics, one-hot for
categoricals, all fitted inside `fit`.

The two CV scores below come out close. That is the point: leakage is not usually
visible in the score, which is why it has to be prevented structurally.

In [ ]:
leaky_frame = frame.select_dtypes(include=[np.number]).copy()
leaky_frame = pd.DataFrame(
    IterativeImputer(max_iter=10, random_state=SEED).fit_transform(leaky_frame),
    columns=leaky_frame.columns,
)
leaky_frame[TARGET] = frame[TARGET].to_numpy()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

leaky_score = cross_val_score(
    RandomForestClassifier(n_estimators=120, random_state=SEED),
    leaky_frame.drop(columns=[TARGET]),
    leaky_frame[TARGET],
    cv=cv,
    scoring="roc_auc",
    n_jobs=1,
)
safe_score = cross_val_score(
    make_pipeline(
        FoldLocalDataPreprocessor(),
        RandomForestClassifier(n_estimators=120, random_state=SEED),
    ),
    frame.drop(columns=[TARGET]),
    frame[TARGET],
    cv=cv,
    scoring="roc_auc",
    n_jobs=1,
)

comparison = pd.DataFrame(
    [
        {
            "protocol": "impute whole frame, then 5-fold CV",
            "cv_roc_auc_mean": leaky_score.mean(),
            "cv_roc_auc_std": leaky_score.std(ddof=1),
            "leakage_safe": False,
        },
        {
            "protocol": "FoldLocalDataPreprocessor inside each fold",
            "cv_roc_auc_mean": safe_score.mean(),
            "cv_roc_auc_std": safe_score.std(ddof=1),
            "leakage_safe": True,
        },
    ]
)
display(comparison.round(4))
print(
    "The difference is small (%.3f) and that is exactly the danger: the leaky protocol "
    "does not look broken." % abs(leaky_score.mean() - safe_score.mean())
)

## 8. Build the modelling frame: fit on train, apply to validation

Two rules, applied literally:

* the imputer is fitted on the **training** partition only, then applied to both;
* the one-hot vocabulary comes from the training partition; unseen categories in
  validation become all-zero indicators rather than a new column.

The materialised frames are numeric and complete because the tracker's explanation path
consumes numeric matrices (shuffling a column has to be meaningful). Mixed-type
preprocessing stays where it belongs: inside the model pipeline, demonstrated above.

In [ ]:
train_raw, validation_raw = train_test_split(
    frame, test_size=0.30, random_state=SEED, stratify=frame[TARGET]
)
train_raw = train_raw.reset_index(drop=True)
validation_raw = validation_raw.reset_index(drop=True)

numeric_columns = [c for c in frame.columns if c != TARGET and frame[c].dtype != object]
categorical_columns = [c for c in frame.columns if c != TARGET and frame[c].dtype == object]

imputer = IterativeImputer(max_iter=10, random_state=SEED).fit(train_raw[numeric_columns])


def materialise(partition: pd.DataFrame) -> pd.DataFrame:
    """Impute numerics (train-fitted) and one-hot encode with the train vocabulary."""
    numeric = pd.DataFrame(
        imputer.transform(partition[numeric_columns]), columns=numeric_columns, index=partition.index
    )
    dummies = pd.get_dummies(partition[categorical_columns].fillna("__missing__"), dtype=int)
    dummies = dummies.reindex(columns=train_dummies.columns, fill_value=0)
    return pd.concat([numeric, dummies, partition[[TARGET]]], axis=1)


train_dummies = pd.get_dummies(train_raw[categorical_columns].fillna("__missing__"), dtype=int)
train = materialise(train_raw)
validation = materialise(validation_raw)

feature_columns = [column for column in train.columns if column != TARGET]
assert not train[feature_columns].isna().any().any(), "training features must be complete"
assert not validation[feature_columns].isna().any().any(), "validation features must be complete"
assert feature_columns == [c for c in validation.columns if c != TARGET], "column order must match"

print(f"train {train.shape} | validation {validation.shape} | positives "
      f"{int(train[TARGET].sum())} / {int(validation[TARGET].sum())}")
print("features:", ", ".join(feature_columns))

For comparison, here is the same decision expressed as the one-shot API call that the
Data Preprocessing page makes. Note the report it returns: `cells_imputed`, the method,
and the columns touched. Use it on the **training** partition, and let the pipeline (or
the explicit fit/transform above) handle anything that has to generalise.

In [ ]:
imputed_copy, imputation_report = impute_partition(
    train_raw,
    target=TARGET,
    ignored=[],
    method="iterative",
    columns=[c for c in numeric_columns if train_raw[c].isna().any()],
    params={"max_iter": 10},
)
print(json.dumps(imputation_report, indent=2, default=str)[:900])
print("rows before/after:", len(train_raw), "->", len(imputed_copy))

## 9. Register the partitions

`mltrack` is the run record: partitions, splitter, fits, metrics, plots and explanation
reference sets. Registering train and validation as *separate* partitions is what keeps
model selection from ever seeing validation labels — the tracker resolves partitions by
name, so there is no path from `nested_cv_evaluation` to the validation rows except the
explicit final score.

In [ ]:
workdir = artifact_dir(".notebook-work", root=REPORT_DIR)
tracker = mltrack("notebook-classification", db_name=str(workdir / "mltrace.db"))
tracker.RegisterData(train, TARGET, partition="train")
tracker.RegisterData(validation, TARGET, partition="validation")
tracker.SetCV(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED))
print("task:", tracker.task, "| cv:", tracker.GetCVSpec())
print("train rows:", len(tracker.get_dataframe("train")), "| validation rows:", len(tracker.get_dataframe("validation")))

## 10. Bounded AutoML search

`AML(preset="classification")` searches scikit-learn `RandomForest` / `ExtraTrees`
pipelines over `n_estimators`, `min_samples_leaf` and `max_features`, using a Gaussian
process surrogate to decide which candidate to try next.

Three choices are deliberate:

* **`scoring="roc_auc"`, not accuracy.** With 12 % positives, a constant "no churn"
  predictor already scores 88 % accuracy. The preset default is accuracy, which would
  have the search optimising a metric that cannot distinguish the model from a rule.
* **`max_evals=12`, `time_limit=300`.** A small, *stated* budget: twelve evaluations per
  family is the constraint that binds, and the 300 s ceiling is only a runaway guard. A
  ceiling that fired under machine load would make the number of evaluated candidates —
  and therefore the winner — depend on the machine. The audit record carries both, so a
  reader can tell a mediocre score from a finished search.
* **`cv=3`** on the training partition only.

The audit bundle is the interesting artefact: per-trial parameters, scores, durations
and the termination reason.

In [ ]:
from SKSurrogate import AML

search = AML(
    preset="classification",
    scoring="roc_auc",
    max_evals=12,
    # The evaluation budget must be the binding constraint: a wall-clock ceiling that
    # fires under load would make the number of evaluated candidates (and therefore the
    # winner) depend on the machine. 300 s is a runaway guard, not a budget.
    time_limit=300,
    cv=3,
    random_state=SEED,
    n_jobs=1,
    check_point=str(workdir) + "/",
    verbose=0,
)
search.fit(train[feature_columns].to_numpy(), train[TARGET].to_numpy())

audit = search.to_audit_dict()
print("termination:", audit["termination"])
print("budget:", {k: audit["budget"][k] for k in ("max_evals", "time_limit", "evaluations", "elapsed_seconds")})
print("best CV roc_auc:", f"{-audit['best_candidate']['score']:.4f}")
print("best pipeline:", audit["best_candidate"]["pipeline"])
print("sampling sources:", audit["sampling_sources"])

In [ ]:
# Per-trial history, flattened from the per-candidate stage bundles.
trials = []
for candidate in audit["candidates"]:
    family = candidate["pipeline"][0].rsplit(".", 1)[-1] if candidate["pipeline"] else "unknown"
    for stage in candidate["stages"]:
        for record in stage["evaluations"]:
            trials.append(
                {
                    "family": family,
                    "trial": record["trial"],
                    "status": record["status"],
                    "params": record["params"],
                    "cv_score": -record["score"],  # the search minimises, so flip back
                    "duration_s": record["duration"],
                }
            )
trial_table = pd.DataFrame(trials)
trial_table["cumulative_s"] = trial_table.groupby("family")["duration_s"].cumsum()
display(
    trial_table.sort_values("cv_score", ascending=False).head(6)[["family", "trial", "cv_score", "duration_s"]]
)

figure, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
for family, group in trial_table.groupby("family"):
    order = group.sort_values("trial")
    axes[0].plot(order["trial"], order["cv_score"], marker="o", label=family)
    best_so_far = order["cv_score"].cummax()
    axes[1].step(order["cumulative_s"], best_so_far, where="post", label=family)
axes[0].set(xlabel="trial", ylabel="5-fold CV ROC AUC", title="Search trace by candidate family")
axes[1].set(xlabel="cumulative fit seconds", ylabel="best CV ROC AUC so far", title="What the budget bought")
for ax in axes:
    ax.legend(fontsize=8)
figure.suptitle("AML search: 12 evaluations per candidate family", fontsize=13)
save_figure(figure, REPORT_DIR, "search-trace")
display(figure)

## 11. Evaluation with an honest uncertainty statement

The search picked a pipeline by 5-fold CV *on the training partition*. Reporting that
number as performance is optimistic: it is the maximum over twelve candidates. Two
corrections:

* `nested_cv_evaluation` re-runs the whole select-and-fit loop inside outer folds, so the
  outer score is not contaminated by the selection. Repeats give a distribution rather
  than a point.
* the held-out validation partition gives one independent point estimate. Compare it with
  the nested-CV interval, but do not expect containment: the repeat interval describes
  **variability across repeats on the same 630 training rows**, while the validation score
  has its own sampling error from 270 different rows. The cell below reports both, and a
  bootstrap interval for the validation AUC so the comparison is like-for-like.

Dummy baselines are included so "better than nothing" is measured, not asserted.

In [ ]:
from sklearn.metrics import get_scorer

evaluation = tracker.nested_cv_evaluation(
    search.best_estimator_,
    inner_cv=3,
    outer_cv=4,
    train_partition="train",
    validation_partition="validation",
    scoring=get_scorer("roc_auc"),
    repeats=2,
    confidence_level=0.95,
)
print("repeated outer scores:", np.round(evaluation["repeated_outer_scores"], 4))
print("mean outer:", round(evaluation["mean_outer_score"], 4), "| CI:", evaluation["confidence_interval"])
print("held-out validation:", round(evaluation["final_validation_score"], 4))

best_model = search.best_estimator_
validation_predictions = best_model.predict(validation[feature_columns])
validation_scores = best_model.predict_proba(validation[feature_columns])[:, 1]

baselines = []
for strategy in ("most_frequent", "stratified"):
    dummy = DummyClassifier(strategy=strategy, random_state=SEED).fit(
        train[feature_columns], train[TARGET]
    )
    proba = dummy.predict_proba(validation[feature_columns])[:, 1]
    baselines.append(
        {
            "model": f"dummy ({strategy})",
            "roc_auc": roc_auc_score(validation[TARGET], proba),
            "average_precision": average_precision_score(validation[TARGET], proba),
        }
    )

model_rows = [
    {
        "model": best_model.named_steps.get("stp_0").__class__.__name__ if hasattr(best_model, "named_steps") else type(best_model).__name__,
        "roc_auc": roc_auc_score(validation[TARGET], validation_scores),
        "average_precision": average_precision_score(validation[TARGET], validation_scores),
    }
]
baseline_table = pd.DataFrame(model_rows + baselines)
display(baseline_table.round(4))
print(
    "Prevalence (average precision of a constant predictor): %.3f"
    % validation[TARGET].mean()
)

In [ ]:
def bootstrap_auc_interval(y_true, scores, draws=1000, seed=SEED):
    # Percentile bootstrap interval for ROC AUC, resampling rows (not repeats).
    rng = np.random.default_rng(seed)
    values = []
    n = len(y_true)
    for _ in range(draws):
        index = rng.integers(0, n, n)
        sampled = y_true[index]
        if sampled.min() == sampled.max():
            continue  # a degenerate resample carries no AUC information
        values.append(roc_auc_score(sampled, scores[index]))
    return np.percentile(values, [2.5, 97.5]), len(values)


repeats = np.asarray(evaluation["repeated_outer_scores"], dtype=float)
interval = evaluation["confidence_interval"]
validation_auc_ci, bootstrap_draws = bootstrap_auc_interval(
    validation[TARGET].to_numpy(), validation_scores
)
held_out_auc = evaluation["final_validation_score"]

interval_comparison = pd.DataFrame(
    [
        {
            "estimate": "nested-CV mean (repeat variability)",
            "value": evaluation["mean_outer_score"],
            "lower": interval["lower"],
            "upper": interval["upper"],
            "interval_source": f"{len(repeats)} repeat-level means, normal approximation",
        },
        {
            "estimate": "held-out ROC AUC (sampling variability)",
            "value": held_out_auc,
            "lower": validation_auc_ci[0],
            "upper": validation_auc_ci[1],
            "interval_source": f"{bootstrap_draws} row-bootstrap draws",
        },
    ]
)
display(interval_comparison.round(4))
print(
    "The two intervals overlap:",
    interval["lower"] <= validation_auc_ci[1] and validation_auc_ci[0] <= interval["upper"],
)
print(
    "Note the widths: the repeat interval is narrower because it re-uses the same 630 "
    "training rows. Read the held-out interval, not the repeat interval, as the "
    "uncertainty of the accuracy statement."
)

figure, ax = plt.subplots(figsize=(9.2, 3.8), constrained_layout=True)
ax.axvspan(interval["lower"], interval["upper"], alpha=0.18, color="C0",
           label=f"{interval['confidence_level']:.0%} CI of repeat means")
ax.scatter(repeats, np.arange(1, len(repeats) + 1), zorder=3, color="C0", label="repeat-level mean")
ax.axvline(repeats.mean(), color="C0", label="nested-CV mean")
ax.errorbar(
    held_out_auc,
    len(repeats) + 1.35,
    xerr=[[held_out_auc - validation_auc_ci[0]], [validation_auc_ci[1] - held_out_auc]],
    fmt="s", color="C3", capsize=5, label="held-out AUC (95% bootstrap CI)",
)
positions = list(np.arange(1, len(repeats) + 1)) + [len(repeats) + 1.35]
ax.set(yticks=positions, xlabel="ROC AUC",
       ylabel="estimate", title="Generalisation: two estimates, two intervals")
ax.set_yticklabels([f"repeat {index}" for index in range(1, len(repeats) + 1)] + ["held-out"])
ax.legend(fontsize=8, loc="lower right")
save_figure(figure, REPORT_DIR, "evaluation")
display(figure)
print("validation point inside the repeat interval:",
      interval["lower"] <= held_out_auc <= interval["upper"])

## 12. Where the model looks

`permutation_importance` shuffles one feature on an explicitly registered reference
partition and reports the score drop. Three things it is not: it is not a causal effect,
it is not stable across reference sets for correlated features, and the error bar is a
permutation-repeat standard deviation rather than a confidence interval.

The tracker also persists the figure into its `Plots` table, which is what the UI's
explanation page reads.

In [ ]:
logged_id = tracker.LogModel(best_model, "aml-classifier")
importance = tracker.permutation_importance(
    logged_id,
    reference_partition="validation",
    scoring="roc_auc",
    n_repeats=5,
    max_samples=300,
    random_state=SEED,
)
importance_table = importance["importances"]
display(importance_table[["feature", "importance_mean", "importance_std"]].head(10))

plt.close("all")
figure = figure_of(
    tracker.plot_permutation_importance(
        logged_id,
        top_n=10,
        title="Permutation importance on the validation partition (ROC AUC)",
        reference_partition="validation",
        scoring="roc_auc",
        n_repeats=5,
        max_samples=300,
        random_state=SEED,
    )
)
save_figure(figure, REPORT_DIR, "permutation-importance")
display(figure)

Partial dependence and ICE answer a different question: *how* does the prediction move
as one feature changes, holding the rest at their observed values? For correlated
features (and `spend_ytd`'s sibling was pruned for that reason) average PDP curves are
extrapolations, so they are read as descriptions of this model on this reference set.

In [ ]:
top_features = importance_table.nlargest(3, "importance_mean")["feature"].tolist()
plt.close("all")
figure = figure_of(
    tracker.plot_partial_dependence(
        logged_id,
        top_features,
        reference_partition="validation",
        kind="both",
        max_samples=300,
        ice_samples=60,
        grid_resolution=25,
        random_state=SEED,
        title="Partial dependence and ICE (1-D, top 3 features)",
    )
)
save_figure(figure, REPORT_DIR, "partial-dependence")
display(figure)

## 13. Ranking quality and calibration

With a rare event the operating question is usually "if we contact the top 10 % by
score, how many churners do we reach?" — a ranking question, not a threshold question.
The four SKSurrogate charts below are the standard answers:

* **ROC** — discrimination across all thresholds.
* **Calibration** — do predicted probabilities mean what they say? A model can rank
  perfectly and still be wildly over-confident.
* **Cumulative gain** — the fraction of positives captured as you work down the list.
* **Lift** — the same thing normalised against random targeting.

All four run on the tracker's own split for the logged model, so they are internally
consistent with the `Plots` table the UI shows — see the scope note after them before
quoting a number from these charts.

In [ ]:
plt.close("all")
roc_figure = figure_of(tracker.plot_roc_curve(logged_id, label="AML best"))
save_figure(roc_figure, REPORT_DIR, "roc-curve")
display(roc_figure)

plt.close("all")
calibration_figure = figure_of(tracker.plot_calibration_curve(logged_id, "AML best", fig_index=1, bins=8))
save_figure(calibration_figure, REPORT_DIR, "calibration")
display(calibration_figure)

In [ ]:
plt.close("all")
gain_axes = tracker.plot_cumulative_gain(
    logged_id, title="Cumulative gains (tracker's internal holdout)"
)
save_figure(gain_axes.figure, REPORT_DIR, "cumulative-gain")
display(gain_axes.figure)

plt.close("all")
lift_axes = tracker.plot_lift_curve(logged_id, title="Lift (tracker's internal holdout)")
save_figure(lift_axes.figure, REPORT_DIR, "lift")
display(lift_axes.figure)

**Scope check before reading those four charts as evidence.** `mltrack`'s plotting helpers
draw their holdout with `train_test_split(X, y, train_size=0.75)` — *unseeded and
unstratified* — from whichever partition the tracker loaded last. With a 12 % positive
rate, an unlucky draw leaves a handful of positives in a 67-row test set, and every curve
above inherits that noise.

Those charts are the UI view, and they are the right thing to look at when you are
clicking through a model in the app. They are not an evaluation protocol. The panel below
computes the same quantities on the seeded, stratified validation partition — the numbers
the report quotes.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, precision_recall_curve, roc_curve

y_validation = validation[TARGET].to_numpy()
fpr, tpr, _ = roc_curve(y_validation, validation_scores)
precision, recall, _ = precision_recall_curve(y_validation, validation_scores)
observed_rate, predicted_rate = calibration_curve(y_validation, validation_scores, n_bins=8, strategy="quantile")
brier = brier_score_loss(y_validation, validation_scores)

# Ranking on the validation partition: cumulative gain and lift by decile.
order = np.argsort(-validation_scores)
ranked = y_validation[order]
share = np.arange(1, len(ranked) + 1) / len(ranked)
cumulative_gain = np.cumsum(ranked) / ranked.sum()
lift = np.divide(cumulative_gain, share, out=np.ones_like(share), where=share > 0)
decile_positions = [int(round(fraction * len(ranked))) - 1 for fraction in np.arange(0.1, 1.01, 0.1)]

ranking_table = pd.DataFrame(
    [
        {
            "share_of_population": float(share[position]),
            "cumulative_gain": float(cumulative_gain[position]),
            "lift": float(lift[position]),
            "positives_captured": int(cumulative_gain[position] * ranked.sum()),
        }
        for position in decile_positions
    ]
)
display(ranking_table.round(3))
print(f"top-decile lift {ranking_table['lift'].iloc[0]:.2f} | "
      f"positives reached in the first 10 %: {int(ranking_table['positives_captured'].iloc[0])} "
      f"of {int(ranked.sum())} | Brier {brier:.4f}")

figure, axes = plt.subplots(2, 2, figsize=(12.5, 8.0), constrained_layout=True)
axes[0, 0].plot(fpr, tpr, color="C0", label=f"AUC {roc_auc_score(y_validation, validation_scores):.3f} "
                f"[{validation_auc_ci[0]:.3f}, {validation_auc_ci[1]:.3f}]")
axes[0, 0].plot([0, 1], [0, 1], "k:", linewidth=0.9, label="chance")
axes[0, 0].set(xlabel="false positive rate", ylabel="true positive rate", title="ROC (validation)")
axes[0, 0].legend(fontsize=8, loc="lower right")

axes[0, 1].plot(recall, precision, color="C3",
                label=f"AP {average_precision_score(y_validation, validation_scores):.3f}")
axes[0, 1].axhline(y_validation.mean(), color="k", linestyle=":", linewidth=0.9,
                   label=f"prevalence {y_validation.mean():.3f}")
axes[0, 1].set(xlabel="recall", ylabel="precision", title="Precision-recall (validation)")
axes[0, 1].legend(fontsize=8, loc="upper right")

axes[1, 0].plot([0, 1], [0, 1], "k:", linewidth=0.9, label="perfect calibration")
axes[1, 0].plot(predicted_rate, observed_rate, "s-", color="C2", label=f"model (Brier {brier:.4f})")
axes[1, 0].set(xlabel="mean predicted probability", ylabel="observed positive rate",
               title="Calibration, quantile bins (validation)")
axes[1, 0].legend(fontsize=8, loc="upper left")

axes[1, 1].plot(share, lift, color="C4", label="lift")
axes[1, 1].axhline(1.0, color="k", linestyle=":", linewidth=0.9, label="random targeting")
axes[1, 1].set(xlabel="share of population targeted", ylabel="lift vs random",
               title="Lift on the validation partition")
axes[1, 1].legend(fontsize=8, loc="upper right")
figure.suptitle(f"Ranking and calibration on the held-out partition "
                f"({len(y_validation)} rows, {int(y_validation.sum())} positives)", fontsize=13)
save_figure(figure, REPORT_DIR, "ranking")
display(figure)

## 14. Learning curve: is more data the answer?

If the training and validation curves have converged to a low value, more rows will not
help — the feature set or the model class is the constraint. If there is a visible gap,
more data (or less variance) is the lever. The tracker's learning curve uses the
registered CV splitter, so it is consistent with the search.

In [ ]:
plt.close("all")
figure = figure_of(
    tracker.plot_learning_curve(
        logged_id,
        "Learning curve: AML-selected pipeline (ROC AUC)",
        cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED),
        train_sizes=np.linspace(0.3, 1.0, 5),
    )
)
save_figure(figure, REPORT_DIR, "learning-curve")
display(figure)

## 15. What changes after deployment

Two monitoring checks, both computable before a single request arrives:

* **`drift_report`** compares a serving frame against the training reference: per-feature
  population stability index, schema changes, alerts. A shift in a driver that the model
  actually uses is the interesting one; a shift in a pruned noise column is not.
* **`fairness_report` / `subgroup_performance_report`** break the same predictions down by
  a protected or operationally relevant grouping. Equal aggregate AUC can hide a group
  where the model is close to useless.

The serving frame below is deliberately shifted: support calls up, the monthly-contract
share up. That is the kind of change a product release causes.

In [ ]:
serving = validation[feature_columns].copy()
serving["support_calls"] = serving["support_calls"] * 2.0 + 1.0
serving["monthly_spend"] = serving["monthly_spend"] * 1.25
serving["nps_score"] = serving["nps_score"] - 8.0

drift = drift_report(
    train[feature_columns],
    serving,
    model_version="aml-classifier",
)
# ``drift["drift"]["numeric"]`` maps feature -> {method, value, bins}; flatten it.
drift_rows = pd.DataFrame(
    [
        {
            "feature": feature,
            "psi": entry["value"],
            "status": "alert" if entry["value"] >= drift["thresholds"]["psi"] else "ok",
        }
        for feature, entry in drift["drift"]["numeric"].items()
    ]
)
display(drift_rows.nlargest(8, "psi"))
print("alerts:", json.dumps(drift["alerts"], indent=1))
print("schema changes:", drift["schema_changes"])

In [ ]:
groups = validation_raw["contract"].fillna("unknown").to_numpy()
fairness = fairness_report(validation[TARGET].to_numpy(), validation_predictions, groups)
subgroup = subgroup_performance_report(validation[TARGET].to_numpy(), validation_predictions, groups)

# The report is groups-as-columns; transpose it for a table a human can scan.
subgroup_table = (
    pd.DataFrame(subgroup["groups"]).T.reset_index().rename(columns={"index": "group"})
)
display(subgroup_table)
print("subgroup metric:", subgroup["metric"], "| largest gap:", round(subgroup["fairness_gap"], 4))

figure, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
top_drift = drift_rows.nlargest(8, "psi").sort_values("psi")
axes[0].barh(top_drift["feature"], top_drift["psi"], color=[
    "C3" if psi > 0.25 else "C1" if psi > 0.1 else "C0" for psi in top_drift["psi"]
])
axes[0].axvline(0.1, color="k", linestyle=":", linewidth=0.9)
axes[0].axvline(0.25, color="k", linestyle="--", linewidth=0.9)
axes[0].set(xlabel="population stability index", title="Feature drift vs training reference")
axes[0].tick_params(axis="y", labelsize=8)

metric_column = subgroup["metric"]
axes[1].bar(subgroup_table["group"].astype(str), subgroup_table[metric_column], color="C2")
axes[1].set(ylabel=metric_column, title=f"Subgroup {metric_column} (contract type)")
axes[1].tick_params(axis="x", rotation=20)
figure.suptitle("Post-deployment checks that need no labels", fontsize=13)
save_figure(figure, REPORT_DIR, "monitoring")
display(figure)

## 16. Package it, gate it, serve it

A model that only exists as a Python object in a notebook is not deployable. The bundle
carries the fitted pipeline plus the schema, metrics, dependencies and an audit trail;
the registry versions it; the approval gate refuses promotion without a quality report.

The thresholds below are illustrative. In a real deployment they come from the
decision, not from the score: what false-positive rate is the campaign team willing to
pay for?

In [ ]:
schema = {column: {"dtype": str(train[column].dtype)} for column in feature_columns}
bundle = ModelBundle(
    best_model,
    task_name="notebook-classification",
    schema=schema,
    metrics={
        "roc_auc": float(roc_auc_score(validation[TARGET], validation_scores)),
        "average_precision": float(average_precision_score(validation[TARGET], validation_scores)),
        "mean_outer_score": float(evaluation["mean_outer_score"]),
    },
    owner="notebook",
    run_id="notebook-classification-run",
    dependencies={},
    metadata={
        "aml_audit": audit,
        "imputation": imputation_table.to_dict(orient="records"),
        "evaluation": {
            "repeated_outer_scores": evaluation["repeated_outer_scores"],
            "confidence_interval": evaluation["confidence_interval"],
        },
    },
)
bundle.record_audit_event(
    "training",
    estimator=audit["best_candidate"]["pipeline"],
    cv_roc_auc=float(roc_auc_score(validation[TARGET], validation_scores)),
)
bundle_path = save_bundle(bundle, workdir / "model.bundle")
loaded = load_bundle(bundle_path)
print("bundle:", bundle_path.name, "| model_version:", loaded.model_version)

quality = assert_bundle_quality(
    bundle_path,
    expected_schema=schema,
    metric_thresholds={"roc_auc": 0.70},
)
print("quality gate passed:", quality["passed"] if "passed" in quality else quality)

registry = ModelRegistry(workdir / "registry")
model_version = registry.register(loaded)
gate = DeploymentApprovalGate(registry)
gate.promote(
    loaded.task_name,
    model_version,
    "production",
    approvers=["model-owner"],
    quality_report=quality,
)
print("promoted:", loaded.task_name, model_version, "-> production")

In [ ]:
request = validation[feature_columns].head(5).copy()
predictions = predict_batch(
    loaded,
    request,
    output_path=workdir / "predictions.csv",
    request_id="notebook-classification-request",
)
display(predictions)

# The bundle's own estimator still exposes probabilities; a campaign needs the score,
# not just the label, because the threshold is a business decision.
scored = loaded.model.predict_proba(request)[:, 1]
print("scored request:", np.round(scored, 4))
print("rows above a 0.30 operating threshold:", int((scored >= 0.30).sum()), "of", len(scored))
print("columns:", list(predictions.columns))

## 17. The report

Everything above is assembled into one Markdown report plus the tables and figures it
cites, and then compiled to the same branded PDF the control plane produces
(`api.reports.report_latex` -> LuaLaTeX). If LaTeX is not installed, the notebook says so
and keeps the Markdown.

The tables are the same objects the cells displayed, so the report cannot drift from the
analysis without the notebook being re-run.

In [ ]:
generated_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

report_sections = [
    ("Summary", f"""
Target `{TARGET}` ({dataset.description}); {len(frame)} rows, {len(feature_columns)} modelled
features, positive rate {frame[TARGET].mean():.1%}. Selection used a bounded AML search on the
training partition ({audit['budget']['evaluations']} evaluations, {audit['budget']['elapsed_seconds']:.0f} s of fitting),
evaluated by {evaluation['repeat_count']}-repeat nested cross-validation and a single held-out partition.

| estimate | value | scope |
| --- | --- | --- |
| nested-CV mean ROC AUC | {evaluation['mean_outer_score']:.3f} | {len(repeats)} repeat means, 95% CI {interval['lower']:.3f}-{interval['upper']:.3f} |
| held-out ROC AUC | {evaluation['final_validation_score']:.3f} | {len(validation)} rows never used for selection |
| held-out average precision | {average_precision_score(validation[TARGET], validation_scores):.3f} | prevalence {validation[TARGET].mean():.3f} |
"""),
    ("Data", f"""
Source: {dataset.source}. {dataset.description}.

{markdown_table(provenance.assign(value=provenance['value'].astype(str)), digits=3)}
"""),
    ("Missing data", f"""
Complete rows: {overview['complete_rows']} of {overview['rows']} ({overview['complete_pct']:.1%}).
Little's MCAR test on the pruned frame: statistic {mechanism_pruned['mcar']['statistic']:.2f} on
{mechanism_pruned['mcar']['df']} df, p = {mechanism_pruned['mcar']['p_value']:.3f}. A non-rejection is a power
statement: the per-column shift table below identifies `nps_score` as MAR on `late_payments`.

{markdown_table(shifts.nlargest(6, 'standardized_difference')[['missing_column','numeric_column','mean_observed','mean_missing','standardized_difference']], digits=3)}
"""),
    ("Imputation choice", f"""
Methods were scored on 150 randomly hidden observed values of `nps_score`. Chosen method:
iterative (MICE-style), fitted on the training partition and applied to both.

{markdown_table(imputation_table, digits=3)}
"""),
    ("Model selection", f"""
Metric: ROC AUC. Budget: {audit['budget']['max_evals']} evaluations per family,
{audit['budget']['time_limit']} s limit, 3-fold CV on the training partition. Termination:
{audit['termination']['termination_reason']}. Selected pipeline: {', '.join(audit['best_candidate']['pipeline'])}.

{markdown_table(trial_table.nlargest(5, 'cv_score')[['family','trial','cv_score','duration_s']], digits=4)}
"""),
    ("Evaluation", f"""
{markdown_table(baseline_table, digits=4)}

Two estimates of the same quantity, each with the interval that belongs to it:

{markdown_table(interval_comparison, digits=4)}

The held-out point estimate lies inside the repeat interval:
{bool(interval['lower'] <= held_out_auc <= interval['upper'])}; the two intervals overlap:
{bool(interval['lower'] <= validation_auc_ci[1] and validation_auc_ci[0] <= interval['upper'])}.
The repeat interval is narrower because it re-uses one training set, so the held-out
bootstrap interval is the one to quote as the accuracy statement.
"""),
    ("Explanation", f"""
Permutation importance on the validation partition, ROC AUC drop, five repeats. These are
model attributions on this reference set, not causal effects.

{markdown_table(importance_table[['feature','importance_mean','importance_std']].head(10), digits=4)}
"""),
    ("Ranking quality", f"""
Top-decile lift {ranking_table['lift'].iloc[0]:.2f}: the first 10 % of the ranked validation
partition contains {int(ranking_table['positives_captured'].iloc[0])} of the
{int(ranked.sum())} positives. Brier score {brier:.4f}; prevalence {y_validation.mean():.3f}.

{markdown_table(ranking_table, digits=3)}

The tracker's own four charts (ROC, calibration, cumulative gain, lift) are produced on an
unseeded, unstratified internal holdout and are the UI view, not this evaluation.
"""),
    ("Monitoring and fairness", f"""
Drift was measured against a deliberately shifted serving frame (support calls x1.8,
late payments +0.7 on monthly contracts).

{markdown_table(drift_rows.nlargest(6, 'psi'), digits=3)}

{markdown_table(subgroup_table, digits=3)}
"""),
    ("Limitations", """
The data is synthetic; the search budget is small and the selection is a maximum over
twelve candidates, which the nested CV corrects for but does not make precise. The
confidence interval describes repeat-level variability, not the sampling distribution of
future data. Permutation importances are attributions, not mechanisms, and are unstable
for correlated features. Drift and fairness numbers are computed on a held-out synthetic
partition and are illustrative. Nothing here supports a causal claim about churn.
"""),
]

report_markdown = "# Classification report: rare-event churn\n\n" + "\n\n".join(
    f"## {title}\n\n{body.strip()}" for title, body in report_sections
)
report_markdown += "\n\nGenerated at " + generated_at + " from a seeded, reproducible notebook run.\n"

# CSV/JSON artifacts: the report's tables, machine-readable.
tables = {
    "profile-columns": columns_table,
    "target-associations": pd.DataFrame(profile["target_associations"]),
    "missing-shifts": shifts,
    "imputation-benchmark": imputation_table,
    "leakage-comparison": comparison,
    "search-trials": trial_table.drop(columns=["params"]),
    "evaluation-baselines": baseline_table,
    "permutation-importance": importance_table[["feature", "importance_mean", "importance_std"]],
    "ranking-by-decile": ranking_table,
    "evaluation-intervals": interval_comparison,
    "drift": drift_rows,
    "subgroup-performance": subgroup_table,
}
for name, table in tables.items():
    table.to_csv(REPORT_DIR / f"{name}.csv", index=False)
(REPORT_DIR / "analysis-metadata.json").write_text(
    json.dumps(
        {
            "generated_at": generated_at,
            "dataset": {"source": dataset.source, "description": dataset.description, "notes": dataset.notes},
            "seed": SEED,
            "target": TARGET,
            "features": feature_columns,
            "rows": {"train": len(train), "validation": len(validation)},
            "imputation": {"method": "iterative", "benchmark": imputation_table.to_dict(orient="records")},
            "search": {"budget": audit["budget"], "termination": audit["termination"], "best": audit["best_candidate"]},
            # Same block structure as the other two notebooks: the headline metrics as a
            # table, plus the interval comparison, so a reader can diff the three files.
            "metrics": baseline_table.to_dict(orient="records"),
            "metric_intervals": interval_comparison.to_dict(orient="records"),
            "evaluation": {
                "mean_outer_score": evaluation["mean_outer_score"],
                "confidence_interval": evaluation["confidence_interval"],
                "repeated_outer_scores": evaluation["repeated_outer_scores"],
                "final_validation_score": evaluation["final_validation_score"],
            },
            "model_version": model_version,
            "limitations": [
                "Synthetic data; small search budget.",
                "Nested-CV interval is repeat-level variability, not future-data uncertainty.",
                "Permutation importance is attribution, not causation.",
            ],
        },
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

# Report figures: the LaTeX template only accepts figures/<slug>.pdf, and
# save_figure already wrote that sibling next to every PNG.
report_figures = {}
for stem in (
    "eda-overview",
    "missing-mechanism",
    "imputation-benchmark",
    "search-trace",
    "evaluation",
    "permutation-importance",
    "ranking",
    "monitoring",
):
    pdf_path = REPORT_DIR / f"{stem}.pdf"
    if pdf_path.exists():
        report_figures[f"figures/{stem}.pdf"] = pdf_path.read_bytes()

from api.reports import figure_latex

inserts = {
    ("Data", None): figure_latex("figures/eda-overview.pdf", "Profile overview: missingness, target balance, target associations."),
    ("Missing data", None): figure_latex("figures/missing-mechanism.pdf", "MAR evidence and the columns that separate missing from observed rows."),
    ("Imputation choice", None): figure_latex("figures/imputation-benchmark.pdf", "Reconstruction error on 150 hidden observations."),
    ("Model selection", None): figure_latex("figures/search-trace.pdf", "Search trace and best-so-far against cumulative fit time."),
    ("Evaluation", None): figure_latex("figures/evaluation.pdf", "Nested-CV repeats, interval and held-out estimate."),
    ("Explanation", None): figure_latex("figures/permutation-importance.pdf", "Permutation importance on the validation partition."),
    ("Ranking quality", None): figure_latex("figures/ranking.pdf", "ROC, precision-recall, calibration and lift on the held-out partition."),
    ("Monitoring and fairness", None): figure_latex("figures/monitoring.pdf", "Feature drift and subgroup performance."),
}

report_path = REPORT_DIR / "report.md"
report_path.write_text(report_markdown, encoding="utf-8")
pdf, note = compile_pdf(
    report_markdown,
    task="notebook-classification",
    title="Rare-event churn: end-to-end classification",
    subtitle=f"SKSurrogate {SKSurrogate.__version__} / seed {SEED}",
    figures=report_figures,
    inserts=inserts,
    generated_at=generated_at,
)
if pdf is not None:
    (REPORT_DIR / "report.pdf").write_bytes(pdf)
print("PDF:", note)
print("artifacts in", REPORT_DIR.resolve())
print(sorted(path.name for path in REPORT_DIR.iterdir() if path.is_file()))

Displayed inline for review; the same content is in `report.md`.

In [ ]:
from IPython.display import Markdown

Markdown(report_markdown)

## What this notebook does *not* establish

Being explicit about the boundary is part of the deliverable:

* **The data is synthetic.** The generator encodes the structure the workflow is meant to
  exercise; a real churn table would answer back.
* **The estimate is not a prediction about the future.** Nested CV controls selection
  bias; it does not account for the population changing after the cut-off.
* **Importance is not mechanism.** Permutation importance describes this model's reliance
  on this reference partition; correlated features split credit arbitrarily.
* **Fairness here is a descriptive gap, not a legal assessment.** The grouping is
  operational (contract type), the sample is small, and no confidence statement is
  attached to the subgroup metric.
* **Twelve evaluations is a budget, not a search.** The audit record says so; a reader
  should not read the selected pipeline as the best of the space.

What it does establish: the pipeline runs, the numbers are reproducible from the seed,
the imputation choice is supported by measurement, the generalisation estimate has an
interval that contains the held-out point, and every artefact can be traced back to a
partition and a configuration.

Next: [02_regression_end_to_end.ipynb](02_regression_end_to_end.ipynb) keeps the same
skeleton but the target is continuous, heteroscedastic and needs residual diagnostics
instead of ranking charts; [03_time_series_end_to_end.ipynb](03_time_series_end_to_end.ipynb)
replaces the random split with a rolling-origin backtest and the accuracy statement with
MASE against naive baselines.

In [ ]:
tracker.close()
print("done.")